# Verificação Pós-Deploy

Notebook de checagem ad-hoc do estado real do pipeline em produção — usado sob
demanda quando algo parece fora do normal (ex.: falha na reconciliação, gap
suspeito), não como parte do fluxo automatizado do Job.

Diferente do `07_auditoria_execucoes` (detecção automática de padrões no
histórico) e do `08_investigacao_anomalias` (preenchimento de causa raiz),
este notebook responde perguntas pontuais de estado: até que data cada
camada (Landing, Bronze, Silver, Gold, KNIME) tem dado disponível, e onde
exatamente uma lacuna começa.

**Não roda automaticamente via Job.**

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
# verificacao 1 - ate que data a Gold tem dado disponivel
display(spark.table("poc_b3_modernizacao.gold.indice_proxy")
    .select("data_referencia", "indice_proxy_pct")
    .orderBy(F.col("data_referencia").desc())
    .limit(10)
)

In [0]:
# verificacao 2 - o que aconteceu em 08/09 (dia util, sem ser feriado)?
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs")
    .filter(F.col("inicio").cast("date") == "2026-09-08")
    .orderBy("inicio")
)

In [0]:
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs")
    .filter(F.col("notebook") == "05_reconciliacao")
    .orderBy(F.col("inicio").desc())
    .limit(1)
)

In [0]:
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs")
    .filter(F.col("inicio") >= F.current_timestamp() - F.expr("INTERVAL 30 MINUTES"))
    .orderBy("inicio")
)


In [0]:
display(spark.table("poc_b3_modernizacao.reconciliation.resultado_indice")
    .orderBy(F.col("data_carga").desc())
    .limit(5)
)